In [ ]:
import glob
import numpy as np
import os
import sys

# add parent folder (production) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bam import MultiBAMv3

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

In [ ]:
input_row = 512
input_col = 33

second_layer = 1024
third_layer = 256

sf = 9
eta=1e-5
num_epochs=10
batch_size=32

In [ ]:

dataset_name_folder = f'dataset_sf{sf}_{input_row}x{input_col}' 
input_layer = input_row * input_col
layers = [input_layer, second_layer, third_layer]  # Ex : compress 3840 → 1024 → 256

GEENRATE_ = True #### Secure accidently running

################### Load all .npy files ########################################
print("LOAD DATASET")
files = glob.glob(f'{dataset_name_folder}/*.npy')
data_list = [np.load(f) for f in files]
# Flatten each spectrogram to 1D (256*15 = 3840)
X = np.array([d.flatten() for d in data_list])  # shape: (num_samples, 3840)
print(X.shape)
################### Load all .npy files ########################################

multi_bam = MultiBAMv3(layers_dims=layers, eta=eta)

if (GEENRATE_):
    
    folder_path = f'weight_{input_layer}_{second_layer}_{third_layer}'

    # Check if folder exists, if not create it
    check_and_make_folder(folder_path)
   
    layer_losses = multi_bam.train(X, num_epochs=num_epochs, batch_size=batch_size)
    
    for i, bam in enumerate(multi_bam.bams):
        np.save(f"{folder_path}/weights_layer_{i}.npy", bam.W)

def load_weight(input_layer,sec,third): 
    ## HOW TO LOAD WEIGHT
    layers = [input_layer, sec, third] # <-- must match training

    multi_bam = MultiBAMv3(layers_dims=layers, eta=eta)
    for i, bam in enumerate(multi_bam.bams):
        bam.W = np.load(f"weight/weights_layer_{i}.npy")
